# Classify Job Postings

Run `3-analyse-aie-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes **all** classified jobs (including non-AI engineering ones) to `jobs/2-classified-jobs.jsonl`. Subsequent notebooks filter by `is_ai_engineering_role` to work with only the accepted roles.

In [1]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

### Load scraped jobs from file

In [2]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

if scraped_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

Scraped jobs JSONL file loaded successfully with 139 entries!


### Define the prompt

We define the instructions that tell the model what counts as an AI engineering role.

In [3]:
client = OpenAI()

instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models or in other words integrating them into products.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- MLOps and platform engineering are not AI engineering, as they focus on infrastructure and tooling rather than building AI-powered features.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

### Define the schema

We tell the model to return structured output with a boolean decision and a short reason.

In [4]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

### Make the LLM calls

We now ask the model to classify each job posting.

In [5]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]

    print(f"Screening job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\n\nDescription:\n{description}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = result["is_ai_engineering_role"]
    reason = result["reason"].strip()

    results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

Screening job 1/139: AI Software Engineer
Screening job 2/139: AI Software Engineer-Senior
Screening job 3/139: Senior AI/ML Engineer - Engineering Excellence (Full Stack Developer) Vice President
Screening job 4/139: Sr. Gen AI Engineer (USC/GC) W2
Screening job 5/139: Principal AI Engineer
Screening job 6/139: Principal Engineer, Agentic AI
Screening job 7/139: Java AI Engineer | Charlotte, NC, USA (3 days onsite) | Contract W2 only
Screening job 8/139: AI Engineer – Optical Design
Screening job 9/139: Senior AI Engineer
Screening job 10/139: Senior Adversarial AI Engineer
Screening job 11/139: Data and AI Engineer
Screening job 12/139: Sr. Data and AI Engineer
Screening job 13/139: Enterprise Automation & AI Engineer
Screening job 14/139: AI Engineer (Hybrid)
Screening job 15/139: VP, Agentic AI Engineering
Screening job 16/139: Senior AI Engineer (USA - Remote)
Screening job 17/139: Junior AI Engineer
Screening job 18/139: Founding AI Engineer
Screening job 19/139: AI Engineer, Voi

### Combine and save the results

We join the model output back to the scraped jobs and save the accepted jobs

In [6]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

results_df = pd.DataFrame(results)
screened_jobs = pd.concat([jobs_df, results_df], axis=1)

non_ai_engineer_count = int((~screened_jobs["is_ai_engineering_role"]).sum())
screened_jobs.to_json(
    classified_jobs_path, orient="records", lines=True, force_ascii=False
)

### Display the results

In [7]:
ai_engineer_count = int(screened_jobs["is_ai_engineering_role"].sum())

print(f"Saved classified jobs to: {classified_jobs_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {ai_engineer_count}")
print(f"Jobs classified as not AI engineering or unclear: {non_ai_engineer_count}")

results_to_show = screened_jobs[
    ["title", "is_ai_engineering_role", "llm_reason", "job_url"]
].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

Saved classified jobs to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-what-is-ai-engineering/jobs/2-classified-jobs.jsonl
Jobs screened by LLM: 139
Jobs classified as AI engineering roles: 103
Jobs classified as not AI engineering or unclear: 36


title,is_ai_engineering_role,llm_reason,job_url
AI Software Engineer,True,"The role is centered on building AI-powered software features on top of foundation models/LLMs: LLM orchestration frameworks, RAG, fine-tuning LLMs, data preparation for LLM consumption, and integrating AI tools into mission systems. This is AI engineering rather than model research or MLOps.",https://www.indeed.com/viewjob?jk=be2d926a6fa9a23a
AI Software Engineer-Senior,True,"The role is centered on building AI-powered software features using LLMs: LLM orchestration, RAG, data preparation for LLM consumption, fine-tuning, and integrating AI tools into mission systems. It is not primarily model research or MLOps/platform work.",https://www.indeed.com/viewjob?jk=5541ba282fe752b1
Senior AI/ML Engineer - Engineering Excellence (Full Stack Developer) Vice President,True,"The role is primarily about building and integrating GenAI/LLM and agentic AI features into enterprise products and workflows, including RAG, prompt engineering, model integration, and deployment of AI-powered applications. Although it includes DevOps, Java, and platform concerns, the core responsibility is application-level AI engineering rather than model research/training or pure infrastructure work.",https://www.indeed.com/viewjob?jk=9053791eafe28b07
Sr. Gen AI Engineer (USC/GC) W2,True,"The role’s core responsibility is building product and production applications on top of foundation models/LLMs (GenAI apps, OpenAI/LangChain/Hugging Face, agentic workflows, multimodal solutions). That fits AI engineering rather than traditional ML, MLOps, or platform work.",https://www.indeed.com/viewjob?jk=7952ffbf91ec53d6
Principal AI Engineer,False,"This is primarily an AI/platform engineering role focused on building infrastructure, tools, APIs, CI/CD, Kubernetes, and cloud services for agentic AI applications. It emphasizes platform engineering and MLOps-style responsibilities more than building end-user AI product features on top of foundation models, so it is not a clear AI engineering role by the stated definition.",https://www.indeed.com/viewjob?jk=f38b13ef5bd2358a
"Principal Engineer, Agentic AI",True,"The role’s core responsibility is building production AI applications and agents on top of foundation models (Copilot Studio, Azure AI Foundry, RAG, orchestration, integrations, evaluation, and governance). It is not primarily model training, MLOps, or traditional ML research.",https://www.indeed.com/viewjob?jk=d53b0d21f106bf44
"Java AI Engineer | Charlotte, NC, USA (3 days onsite) | Contract W2 only",True,"The role appears to be building conversational AI/product features on top of existing LLMs and agentic frameworks (Dialogflow CX, LLMs, prompt engineering, ADK), rather than training or tuning models. The core software stack is Java/Python backend integration around AI capabilities, which fits AI engineering.",https://www.indeed.com/viewjob?jk=f2e036639932b33a
AI Engineer – Optical Design,True,"The role is centered on building and iterating on an agentic/LLM-integrated system, running experiments, debugging tool loops and multi-turn state, and evolving toward production architecture for AI-powered features. That fits AI engineering (applications on foundation models), not model training/MLOps.",https://www.indeed.com/viewjob?jk=836a713564071627
Senior AI Engineer,True,"The role is primarily building and maintaining an enterprise conversational AI product on top of foundation models (Azure OpenAI, prompt engineering, RAG, bot development, natural language interfaces, business integrations). It is focused on applying existing LLMs to product features rather than training models or doing MLOps/platform-only work.",https://www.indeed.com/viewjob?jk=16933d29b84666ae
Senior Adversarial AI Engineer,False,"This role is primarily AI security/adversarial analysis and evaluation, with research into defenses, threat analysis, red teaming, and integration work. It is not mainly about building product fe